# Knowledge Graph Pipeline1: Proof-of-Concept (without ODKE+)

This notebook implements a pipeline for financial knowledge extraction using FNSPID dataset.

In [123]:
from huggingface_hub import hf_hub_download
import pandas as pd
import os
# from google.colab import userdata
from pydantic import BaseModel, Field, ValidationError
from typing import Optional, List, Type, TypeVar, Generic
import enum
from datetime import datetime
import requests
from bs4 import BeautifulSoup
from google import genai
from google.genai import types
import json
from neo4j import GraphDatabase, Session, Transaction
from dotenv import load_dotenv
from langchain_neo4j import Neo4jGraph
from pyvis.network import Network
from IPython.display import FileLink
from typing import Dict, List, Any
import csv


In [111]:
def join_string(item):
    Date, Article_title, Stock_symbol, Url, Lexrank_summary = item
    final_string = ""
    
    # Check if column has a unique value
    if pd.notna(Date):
        final_string += f"Date: {Date}"
    
    if pd.notna(Article_title):
        if final_string:
            final_string += " | "
        final_string += f"Article Title: {Article_title}"
    
    if pd.notna(Stock_symbol):
        if final_string:
            final_string += " | "
        final_string += f"Stock Symbol: {Stock_symbol}"
    
    if pd.notna(Url):
        if final_string:
            final_string += " | "
        final_string += f"Url: {Url}"
    
    if pd.notna(Lexrank_summary):
        if final_string:
            final_string += " | "
        final_string += f"Summary: {Lexrank_summary}"
    
    return final_string

def filtered_text(file_path):
    df = pd.read_csv(
        file_path,
        dtype=str,
        low_memory=False,
        encoding='utf-8',
        on_bad_lines='skip'  # Skip problematic lines
    )
    
    # DataFrame processing
    df['Date'] = pd.to_datetime(df['Date'])
    date_time = input("Date time to filter (e.g., 2023-12-01 00:00:00+00:00): ")
    ticker_symbol = input("Ticker symbol to filter (e.g., AAPL, AMZN, GOOGL): ")
    df_filtered = df[(df['Date'].dt.date == pd.to_datetime(date_time).date()) & (df['Stock_symbol'] == ticker_symbol)]
    
    # Create the 'information' column
    df_filtered['Information'] = df_filtered[
        ['Date', 'Article_title', 'Stock_symbol', 'Url', 'Lexrank_summary']
    ].apply(join_string, axis=1)
    
    # Group all information and return as text
    df_grouped = df_filtered.groupby('Date')['Information'].apply(lambda x: '\n'.join(x)).reset_index()
    sample = df_grouped.head().iloc[0]['Information']
    
    return sample

In [ ]:
def extract_entities_and_relationship(text):
    prompt = f"""
# Role
You are an expert Financial Data Analyst and Knowledge Graph Engineer. Your task is to analyze financial news summaries from Nasdaq and extract a structured knowledge graph in JSON format.

# Task
You will be provided with a raw text block containing metadata (Date, Title, Ticker, URL) and a Summary. You must extract semantic triplets representing the relationships between entities mentioned in the text.

# Ontology (Strict Compliance Required)
You must strictly use ONLY the following Entity Types and Relationship Types. Do not invent new types.

## 1. Entity Types (24 Categories)
**Core Business Entities**
- ORG: Filing Company (the main subject of the news/ticker)
- COMP: External companies (competitors, suppliers, customers, partners)
- SEGMENT: Internal business divisions (e.g., Cloud segment)
- PERSON: Key individuals (Executives, Board members)

**Geographic & Regulatory**
- GPE: Geographic entities (Countries, cities)
- ORG_GOV: Government bodies (e.g., US Gov)
- ORG_REG: Regulatory bodies (SEC, Fed, ECB)

**Financial & Market**
- FIN_INST: Financial instruments (Stocks, bonds, options)
- FIN_MARKET: Market indices (S&P 500, Nasdaq)
- FIN_METRIC: Financial metrics (Revenue, EPS, Share Price)
- ECON_IND: Economic indicators (Inflation, GDP)

**Products & Operations**
- PRODUCT: Products or services (iPhone, AWS)
- CONCEPT: Abstract concepts (AI, Digital Transformation)
- RAW_MATERIAL: Essential materials (Lithium, Oil)
- LOGISTICS: Supply chain entities (Ports, Shipping lanes)

**Risk & Compliance**
- RISK_FACTOR: Documented risks (Recession risk, Cyber attacks)
- LITIGATION: Legal disputes, lawsuits
- REGULATORY_REQUIREMENT: Specific regulations (GDPR, Basel III)
- ACCOUNTING_POLICY: Policies (Revenue recognition)

**Strategic & ESG**
- EVENT: Material events (Earnings call, M&A, Pandemic)
- SECTOR: Industries (Technology, Healthcare)
- ESG_TOPIC: ESG themes (Carbon, DEI)
- MACRO_CONDITION: Economic trends (Recession, Labor shortage)
- COMMENTARY: Management statements/guidance

## 2. Relationship Types (27 Categories)
**Ownership & Control**
- Has_Stake_In
- Regulates
- Operates_In

**Business Activities**
- Announces
- Introduces
- Produces
- Invests_In
- Partners_With
- Supplies

**Financial Impact**
- Impacts
- Positively_Impacts
- Negatively_Impacts
- Increases
- Decreases
- Affects_Stock

**Risk & Events**
- Involved_In
- Impacted_By
- Faces
- Depends_On

**Market & Reporting**
- Discloses
- Guides_On
- Complies_With
- Subject_To

**Additional Relations**
- Related_To
- Member_Of
- Causes_Shortage_Of
- Stock_Decline_Due_To
- Stock_Rise_Due_To
- Market_Reacts_To

# Extraction Rules
1. **Metadata Extraction:** Parse the input header to extract the global `date`, `ticker`, and `source_url`. Apply these to every triplet found in that text.
2. **Entity Resolution:**
   - If the main ticker (e.g., AAPL) is mentioned, label it as `ORG`.
   - Other companies mentioned (e.g., McDonald's in an Apple article) should be labeled `COMP`.
3. **Granularity:** Extract specific named entities from the text (e.g., "Apple Inc." instead of just "Company").
4. **Source Text:** You must include the exact sentence or phrase where the relationship was found in the `source_text` field.
5. **Preserve ALL numerical figures and dates exactly.**
6. **Keep ALL dates in their original format.**

# Output Format
Return a single JSON list of objects. Do not include markdown formatting (like ```json) outside of the list.

JSON Structure:
[
    {{
        "triplet_id": "UUID or Sequence",
        "entity": "Extracted Head Entity",
        "entity_type": "One of the 24 types",
        "relationship": "One of the 27 types",
        "target": "Extracted Tail Entity",
        "target_type": "One of the 24 types",
        "date": "YYYY-MM-DD",
        "ticker": "Symbol from metadata",
        "source_url": "URL from metadata",
        "source_text": "Evidence text snippet"
    }}
]

# Few-Shot Example

**Input:**
Date: 2023-12-01 | Article Title: Best Blue Chip Stocks? | Stock Symbol: AAPL | Url: https://nasdaq.com/article/123
Summary: Apple Inc. (AAPL) is a multinational technology company that specializes in consumer electronics. Recently, shares of AAPL stock have gained by 9.29% due to holiday sales.

**Output:**
[
    {{
        "triplet_id": "1",
        "entity": "Apple Inc.",
        "entity_type": "ORG",
        "relationship": "Produces",
        "target": "consumer electronics",
        "target_type": "PRODUCT",
        "date": "2023-12-01",
        "ticker": "AAPL",
        "source_url": "https://nasdaq.com/article/123",
        "source_text": "Apple Inc. (AAPL) is a multinational technology company that specializes in consumer electronics."
    }},
    {{
        "triplet_id": "2",
        "entity": "AAPL",
        "entity_type": "ORG",
        "relationship": "Increases",
        "target": "Share Price",
        "target_type": "FIN_METRIC",
        "date": "2023-12-01",
        "ticker": "AAPL",
        "source_url": "https://nasdaq.com/article/123",
        "source_text": "shares of AAPL stock have gained by 9.29%"
    }},
    {{
        "triplet_id": "3",
        "entity": "AAPL",
        "entity_type": "ORG",
        "relationship": "Stock_Rise_Due_To",
        "target": "Holiday Sales",
        "target_type": "EVENT",
        "date": "2023-12-01",
        "ticker": "AAPL",
        "source_url": "https://nasdaq.com/article/123",
        "source_text": "shares of AAPL stock have gained by 9.29% due to holiday sales."
    }}
]

# Input Text for Processing
{text}
"""
    load_dotenv()
    client = genai.Client(api_key=os.getenv('GOOGLE_API_KEY', ''))
    response = client.models.generate_content(
        model='gemini-2.5-flash',
        contents=prompt,
        config=types.GenerateContentConfig(
            temperature=0.1,
            response_mime_type="application/json"
        )
    )
    
    result = json.loads(response.text)
    return result

In [68]:
def clear_kg(kg):
    with kg._driver.session() as session:
        session.run("MATCH (n) DETACH DELETE n")

def add_relationship_to_neo4j(kg, triplets):
    with kg._driver.session() as session:
        for triplet in triplets:
            rel_type = triplet['relationship']
            query = f"MERGE (a:Entity {{name: $source}}) " \
                    f"ON CREATE SET a.type = $source_type " \
                    f"MERGE (b:Entity {{name: $target}}) " \
                    f"ON CREATE SET b.type = $target_type " \
                    f"MERGE (a)-[r:{rel_type}]->(b) " \
                    f"SET r.date = $date, " \
                    f"r.ticker = $ticker, " \
                    f"r.source_url = $source_url, " \
                    f"r.source_text = $source_text"
            try:
                session.run(
                    query,
                    source=triplet['entity'],
                    source_type=triplet['entity_type'],
                    target=triplet['target'],
                    target_type=triplet['target_type'],
                    date=triplet.get('date', 'N/A'),
                    ticker=triplet.get('ticker', 'N/A'),
                    source_url=triplet.get('source_url', ''),
                    source_text=triplet.get('source_text', '')
                )
            except Exception as e:
                print(f"Error adding relationship: {e}")                        

In [70]:
def visualize_kg(graph_wrapper, filename="financial_graph.html"):
    query = """
    MATCH (n:Entity)-[r]->(m:Entity) 
    RETURN n.name AS source, n.type AS source_type,
           m.name AS target, m.type AS target_type,
           type(r) AS rel_type,
           r.date AS date,
           r.ticker AS ticker,
           r.source_url AS source_url
    """
    results = graph_wrapper.query(query)
    
    net = Network(
        height="750px", 
        width="100%", 
        bgcolor="#222222", 
        font_color="white", 
        directed=True
    )
    
    # Configure node and edge appearance
    net.set_options("""
    {
        "nodes": {
            "font": {
                "size": 14,
                "color": "white"
            },
            "size": 25,
            "borderWidth": 2
        },
        "edges": {
            "font": {
                "size": 12,
                "align": "middle",
                "color": "white"
            },
            "arrows": {
                "to": {
                    "enabled": true,
                    "scaleFactor": 0.5
                }
            },
            "smooth": {
                "type": "continuous"
            }
        },
        "physics": {
            "barnesHut": {
                "gravitationalConstant": -30000,
                "centralGravity": 0.3,
                "springLength": 200,
                "springConstant": 0.04
            },
            "minVelocity": 0.75
        }
    }
    """)
    
    # Track nodes to avoid duplicates
    added_nodes = set()
    
    for record in results:
        source_name = record['source']
        target_name = record['target']
        
        # Add source node if not already added
        if source_name not in added_nodes:
            net.add_node(
                source_name, 
                label=f"{source_name}\n[{record['source_type']}]",
                color="#00d4ff", 
                title=f"{source_name} ({record['source_type']})",
                size=30,
                shape='ellipse'
            )
            added_nodes.add(source_name)
        
        # Add target node if not already added
        if target_name not in added_nodes:
            net.add_node(
                target_name, 
                label=f"{target_name}\n[{record['target_type']}]",
                color="#ff6b6b", 
                title=f"{target_name} ({record['target_type']})",
                size=30,
                shape='ellipse'
            )
            added_nodes.add(target_name)
        
        # Create edge with additional metadata
        edge_title = f"{record['rel_type']}"
        if record.get('date') and record['date'] != 'N/A':
            edge_title += f"\nDate: {record['date']}"
        if record.get('ticker') and record['ticker'] != 'N/A':
            edge_title += f"\nTicker: {record['ticker']}"
        if record.get('source_url'):
            edge_title += f"\nSource: {record['source_url']}"
        
        net.add_edge(
            source_name, 
            target_name, 
            label=record['rel_type'],
            title=edge_title,
            color="#888888"
        )
    
    net.save_graph(filename)
    return FileLink(filename)

In [129]:
CYPHER_GENERATION_TEMPLATE = """
# Role
You are an expert in Neo4j Cypher query language and financial knowledge graphs. Your task is to convert natural language questions into precise Cypher queries.

# Knowledge Graph Schema

## Node Structure
- Label: Entity
- Properties:
  - name: string (entity name)
  - type: string (entity type from 24 categories)

## Entity Types (24 Categories)
**Core Business:** ORG, COMP, SEGMENT, PERSON
**Geographic & Regulatory:** GPE, ORG_GOV, ORG_REG
**Financial & Market:** FIN_INST, FIN_MARKET, FIN_METRIC, ECON_IND
**Products & Operations:** PRODUCT, CONCEPT, RAW_MATERIAL, LOGISTICS
**Risk & Compliance:** RISK_FACTOR, LITIGATION, REGULATORY_REQUIREMENT, ACCOUNTING_POLICY
**Strategic & ESG:** EVENT, SECTOR, ESG_TOPIC, MACRO_CONDITION, COMMENTARY

## Relationship Structure
- Dynamic relationship types (27 types): Has_Stake_In, Regulates, Operates_In, Announces, Introduces, Produces, Invests_In, Partners_With, Supplies, Impacts, Positively_Impacts, Negatively_Impacts, Increases, Decreases, Affects_Stock, Involved_In, Impacted_By, Faces, Depends_On, Discloses, Guides_On, Complies_With, Subject_To, Related_To, Member_Of, Causes_Shortage_Of, Stock_Decline_Due_To, Stock_Rise_Due_To, Market_Reacts_To
- Properties:
  - date: string
  - ticker: string
  - source_url: string
  - source_text: string

# CRITICAL: Always Include Source Information
Every query MUST return both source_url and source_text properties from relationships to enable proper verification.
Use this pattern: r.source_url AS SourceURL, r.source_text AS Evidence

# Few-Shot Examples

## Example 1: Direct Entity Query
**Question:** "What products does Apple produce?"
**Analysis:**
- Entity: Apple (ORG)
- Relationship: Produces
- Target Type: PRODUCT

**Cypher Query:**
```cypher
MATCH (org:Entity {{name: 'AAPL'}})-[r:Produces]->(product:Entity)
WHERE product.type = 'PRODUCT'
RETURN org.name AS Company, 
       product.name AS Product, 
       r.date AS Date, 
       r.source_url AS SourceURL,
       r.source_text AS Evidence
```

**Explanation:** Find all entities connected to Apple through Produces relationship where target is a PRODUCT type. Returns source URL and text for verification.

**Expected Output:** List of products Apple produces with supporting evidence and source URLs.

## Example 2: Financial Metric Query
**Question:** "How much did AAPL stock increase?"
**Analysis:**
- Entity: AAPL (ORG)
- Relationship: Increases
- Target Type: FIN_METRIC

**Cypher Query:**
```cypher
MATCH (org:Entity)-[r:Increases]->(metric:Entity)
WHERE org.name CONTAINS 'AAPL' AND metric.type = 'FIN_METRIC'
RETURN org.name AS Entity, 
       metric.name AS Metric, 
       r.date AS Date, 
       r.source_url AS SourceURL,
       r.source_text AS Evidence
ORDER BY r.date DESC
```

**Explanation:** Find financial metrics that increased, filtering by AAPL and metric type. Includes source URL for verification.

**Expected Output:** Percentage increases with dates, evidence text, and source URLs.

## Example 3: Relationship Discovery
**Question:** "Who are Apple's competitors?"
**Analysis:**
- Entity: Apple (ORG/COMP)
- Relationship: COMPETES_WITH or related competitive relationships
- Target Type: COMP/ORG

**Cypher Query:**
```cypher
MATCH (apple:Entity)-[r]-(competitor:Entity)
WHERE apple.name CONTAINS 'Apple' 
  AND (competitor.type = 'COMP' OR competitor.type = 'ORG')
  AND type(r) IN ['COMPETES_WITH', 'Related_To']
RETURN apple.name AS Company, 
       type(r) AS Relationship, 
       competitor.name AS Competitor, 
       r.date AS Date,
       r.source_url AS SourceURL,
       r.source_text AS Evidence
```

**Explanation:** Find entities related to Apple that are companies, checking multiple relationship types. Returns source URLs for verification.

**Expected Output:** List of competitor companies with relationship context and source URLs.

## Example 4: Event Impact Query
**Question:** "What events affected Apple's stock?"
**Analysis:**
- Entity: Apple (ORG)
- Relationship: Stock_Rise_Due_To, Stock_Decline_Due_To, Affects_Stock
- Target Type: EVENT

**Cypher Query:**
```cypher
MATCH (org:Entity)-[r]->(event:Entity)
WHERE org.name CONTAINS 'Apple' 
  AND event.type = 'EVENT'
  AND type(r) IN ['Stock_Rise_Due_To', 'Stock_Decline_Due_To', 'Affects_Stock']
RETURN org.name AS Company, 
       type(r) AS Impact, 
       event.name AS Event, 
       r.date AS Date, 
       r.source_url AS SourceURL,
       r.source_text AS Evidence
ORDER BY r.date DESC
```

**Explanation:** Find all events that had stock impact on Apple, showing the type of impact. Includes source URLs for verification.

**Expected Output:** Events with their impact direction, supporting evidence, and source URLs.

## Example 5: Multi-Hop Query
**Question:** "What products does Apple produce and which events increased their sales?"
**Analysis:**
- Multi-step: Apple -> Produces -> Product, Product -> Increases -> EVENT
- Entity Types: ORG, PRODUCT, EVENT
- Relationships: Produces, Increases, Stock_Rise_Due_To

**Cypher Query:**
```cypher
MATCH (org:Entity {{name: 'Apple Inc.'}})-[r1:Produces]->(product:Entity)-[r2:Increases|Stock_Rise_Due_To]->(target:Entity)
WHERE product.type = 'PRODUCT' AND (target.type = 'EVENT' OR target.type = 'FIN_METRIC')
RETURN org.name AS Company, 
       product.name AS Product, 
       type(r2) AS Relationship, 
       target.name AS Target, 
       r2.date AS Date,
       r2.source_url AS SourceURL,
       r2.source_text AS Evidence
```

**Explanation:** Two-hop query connecting Apple to products to sales increases or events. Returns source information from the final relationship.

**Expected Output:** Products and the events/metrics that increased their performance with source URLs.

## Example 6: Aggregation Query
**Question:** "How many relationships does Apple have in the graph?"
**Analysis:**
- Count query
- Entity: Apple
- All relationships

**Cypher Query:**
```cypher
MATCH (org:Entity)-[r]-(other:Entity)
WHERE org.name CONTAINS 'Apple'
RETURN org.name AS Company, 
       count(r) AS TotalRelationships, 
       collect(DISTINCT type(r)) AS RelationshipTypes,
       collect(DISTINCT r.source_url)[0..5] AS SampleSourceURLs
```

**Explanation:** Count all relationships for Apple and list unique relationship types. Returns sample source URLs for reference.

**Expected Output:** Total count, list of relationship types, and sample source URLs.

## Example 7: Time-Based Query
**Question:** "What happened with Apple in December 2023?"
**Analysis:**
- Entity: Apple
- Time filter: December 2023
- All relationships

**Cypher Query:**
```cypher
MATCH (org:Entity)-[r]-(other:Entity)
WHERE org.name CONTAINS 'Apple' 
  AND r.date CONTAINS '2023-12'
RETURN org.name AS Company, 
       type(r) AS Relationship, 
       other.name AS RelatedEntity, 
       other.type AS EntityType, 
       r.date AS Date, 
       r.source_url AS SourceURL,
       r.source_text AS Evidence
ORDER BY r.date DESC
```

**Explanation:** Find all Apple-related activities in December 2023. Returns source URLs and evidence text for each relationship.

**Expected Output:** All relationships and entities for that time period with source URLs for verification.

# Instructions
1. Analyze the question to identify:
   - Key entities mentioned
   - Entity types involved
   - Relationship types needed
   - Any filters (date, ticker, metric type)
   - Whether aggregation is needed

2. Generate a Cypher query that:
   - Uses proper Neo4j syntax
   - Filters by entity types when relevant
   - Includes date/ticker filters if mentioned
   - **ALWAYS returns r.source_url AS SourceURL and r.source_text AS Evidence**
   - Returns meaningful column names
   - Orders results when appropriate

3. Provide:
   - The Cypher query
   - Brief explanation of the query logic
   - Description of expected output

# MANDATORY RETURN PATTERN
Every query MUST include these columns when relationships are involved:
- r.source_url AS SourceURL
- r.source_text AS Evidence

For multi-hop queries, include source information from the most relevant relationship (typically the final one).

# Output Format
Return ONLY a valid JSON object (no markdown formatting):
{{
    "cypher_query": "MATCH ...",
    "explanation": "Brief explanation of the query logic",
    "expected_output": "Description of what results will show"
}}

# Question to Convert
{question}
"""

def generate_cypher_query(question: str) -> Dict[str, Any]:
  load_dotenv()
  client = genai.Client(api_key=os.getenv('GOOGLE_API_KEY', ''))
  prompt = CYPHER_GENERATION_TEMPLATE.format(question=question)
  try:
    response = client.models.generate_content(
      model = 'gemini-2.5-flash',
      contents=prompt,
      config={'temperature': 0.1, 'response_mime_type': 'application/json'}
    )
    result = json.loads(response.text)
    return result
  
  except Exception as e:
    return {
      'cypher_query': None,
      'explanation': f"Error: {str(e)}",
      'expected_output': None
    }

def execute_cypher_query(graph, cypher_query: str) -> List[Dict[str, Any]]:
  try:
    result = graph.query(cypher_query)
    records = []
    
    for record in result:
      records.append(dict(record))
    
    return records

  except Exception as e:
      print(f"Error executing Cypher query: {e}")
      return []

def query_chatbot(question: str, neo4j_session) -> Dict[str, Any]:
  # Generate Cypher query
  query_info = generate_cypher_query(question)
  if not query_info['cypher_query']:
    return {
      'question': question,
      'cypher_query': None,
      'results': [],
      'answer': 'N/A'
    }
  
  # Execute query
  results = execute_cypher_query(neo4j_session, query_info['cypher_query'])
  
  return {
    'question': question,
    'cypher_query': query_info['cypher_query'],
    'results': results,
  }
  
def query_to_csv(query_response: Dict[str, Any], filename: str = 'query_results.csv'):
  question = query_response.get('question', 'N/A')
  cypher_query = query_response.get('cypher_query', 'N/A')
  results = query_response.get('results', [])
  
  # Check if file exists to determine if we need to write header
  file_exists = os.path.isfile(filename)
  
  with open(filename, 'a', newline='', encoding='utf-8') as f:
      writer = csv.writer(f)
      
      # Write header only if file doesn't exist
      if not file_exists:
          writer.writerow(['Question', 'Cypher Query', 'Result'])
      
      if not results:
          # No results case
          writer.writerow([question, cypher_query, 'No results found'])
      else:
          # Merge all results
          merged_results = []
          for result in results:
              # Check for standard evidence/source fields
              evidence = result.get('Evidence', result.get('source_text', None))
              source_url = result.get('SourceURL', result.get('source_url', None))
              
              # Check for aggregation fields (arrays)
              sample_evidence = result.get('SampleEvidence', None)
              sample_urls = result.get('SampleSourceURLs', None)
              
              if evidence and source_url:
                  # Standard query result
                  merged_results.append(f"{evidence} from sourceURL: {source_url}")
              elif sample_evidence and sample_urls:
                  # Aggregation query with arrays - pair them up
                  for ev, url in zip(sample_evidence, sample_urls):
                      if ev and url:
                          merged_results.append(f"{ev} from sourceURL: {url}")
              else:
                  # Fallback: create summary from all non-standard fields
                  summary_parts = []
                  for key, value in result.items():
                      if key not in ['Evidence', 'SourceURL', 'source_text', 'source_url', 'SampleEvidence', 'SampleSourceURLs']:
                          summary_parts.append(f"{key}: {value}")
                  if summary_parts:
                      merged_results.append(' | '.join(summary_parts))
          
          # Join all results with newline
          final_result = '\n'.join(merged_results) if merged_results else 'No evidence found'
          
          # Write single row with all results
          writer.writerow([question, cypher_query, final_result])

In [130]:
if __name__ == "__main__":
    # 1. Download dataset from huggingface
    # file_path = hf_hub_download(
    #     repo_id="Zihan1004/FNSPID",
    #     filename="Stock_news/nasdaq_exteral_data.csv",
    #     repo_type="dataset"
    # )
    # . Or load nasdaq_100.csv data and filtered from local repo after clone from github
    # file_path = "/Users/anhvu/workspace/conda_env/IDM-2025/nasdaq_100.csv"
    # sample = filtered_text(file_path)
    # . Or open data directly from local repo after clone from github
    # with open('/Users/anhvu/workspace/conda_env/IDM-2025/sample-text', 'r') as f: # hard-coded for POC development
    #     sample = f.read()
        
    # 2. Use LLMs to extract kg entities and relationship
    # triplets = extract_entities_and_relationship(sample)
    triplets = [{'triplet_id': '1', 'entity': 'AAPL', 'entity_type': 'ORG', 'relationship': 'Increases', 'target': 'Share Price by 9.29%', 'target_type': 'FIN_METRIC', 'date': '2023-12-01 00:00:00+00:00', 'ticker': 'AAPL', 'source_url': 'https://www.nasdaq.com/articles/best-blue-chip-stocks-to-invest-in-right-now-2-in-focus', 'source_text': 'shares of AAPL stock have gained by 9.29%.'}, {'triplet_id': '2', 'entity': 'Apple Inc.', 'entity_type': 'ORG', 'relationship': 'Related_To', 'target': 'multinational technology company', 'target_type': 'SECTOR', 'date': '2023-12-01 00:00:00+00:00', 'ticker': 'AAPL', 'source_url': 'https://www.nasdaq.com/articles/best-blue-chip-stocks-to-invest-in-right-now-2-in-focus', 'source_text': 'Apple Inc. (AAPL) is a multinational technology company'}, {'triplet_id': '3', 'entity': 'Apple Inc.', 'entity_type': 'ORG', 'relationship': 'Produces', 'target': 'consumer electronics', 'target_type': 'PRODUCT', 'date': '2023-12-01 00:00:00+00:00', 'ticker': 'AAPL', 'source_url': 'https://www.nasdaq.com/articles/best-blue-chip-stocks-to-invest-in-right-now-2-in-focus', 'source_text': 'Apple Inc. (AAPL) is a multinational technology company that specializes in consumer electronics'}, {'triplet_id': '4', 'entity': 'Apple Inc.', 'entity_type': 'ORG', 'relationship': 'Produces', 'target': 'software', 'target_type': 'PRODUCT', 'date': '2023-12-01 00:00:00+00:00', 'ticker': 'AAPL', 'source_url': 'https://www.nasdaq.com/articles/best-blue-chip-stocks-to-invest-in-right-now-2-in-focus', 'source_text': 'Apple Inc. (AAPL) is a multinational technology company that specializes in consumer electronics, software'}, {'triplet_id': '5', 'entity': 'Apple Inc.', 'entity_type': 'ORG', 'relationship': 'Produces', 'target': 'online services', 'target_type': 'PRODUCT', 'date': '2023-12-01 00:00:00+00:00', 'ticker': 'AAPL', 'source_url': 'https://www.nasdaq.com/articles/best-blue-chip-stocks-to-invest-in-right-now-2-in-focus', 'source_text': 'Apple Inc. (AAPL) is a multinational technology company that specializes in consumer electronics, software, and online services.'}, {'triplet_id': '6', 'entity': 'McDonald’s Corporation', 'entity_type': 'COMP', 'relationship': 'Related_To', 'target': 'Blue Chip Stocks', 'target_type': 'FIN_MARKET', 'date': '2023-12-01 00:00:00+00:00', 'ticker': 'AAPL', 'source_url': 'https://www.nasdaq.com/articles/best-blue-chip-stocks-to-invest-in-right-now-2-in-focus', 'source_text': 'Blue Chip Stocks To Buy [Or Avoid] Today McDonald’s Corporation (NYSE: MCD)'}, {'triplet_id': '7', 'entity': 'Apple Inc.', 'entity_type': 'ORG', 'relationship': 'Related_To', 'target': 'Blue Chip Stocks', 'target_type': 'FIN_MARKET', 'date': '2023-12-01 00:00:00+00:00', 'ticker': 'AAPL', 'source_url': 'https://www.nasdaq.com/articles/best-blue-chip-stocks-to-invest-in-right-now-2-in-focus', 'source_text': 'Apple Inc. (NASDAQ: AAPL)'}, {'triplet_id': '8', 'entity': 'Paramount Global', 'entity_type': 'COMP', 'relationship': 'Increases', 'target': 'Share Price by 7.6%', 'target_type': 'FIN_METRIC', 'date': '2023-12-01 00:00:00+00:00', 'ticker': 'AAPL', 'source_url': 'https://www.nasdaq.com/articles/us-stocks-wall-st-edges-higher-as-powell-comments-bolster-peak-rate-bets', 'source_text': 'Paramount GlobalPARA.O climbed 7.6%'}, {'triplet_id': '9', 'entity': 'Paramount Global', 'entity_type': 'COMP', 'relationship': 'Partners_With', 'target': 'Apple', 'target_type': 'ORG', 'date': '2023-12-01 00:00:00+00:00', 'ticker': 'AAPL', 'source_url': 'https://www.nasdaq.com/articles/us-stocks-wall-st-edges-higher-as-powell-comments-bolster-peak-rate-bets', 'source_text': 'the media company and Apple AAPL.O have discussed bundling their streaming services at a discount.'}, {'triplet_id': '10', 'entity': 'Apple', 'entity_type': 'ORG', 'relationship': 'Partners_With', 'target': 'Paramount Global', 'target_type': 'COMP', 'date': '2023-12-01 00:00:00+00:00', 'ticker': 'AAPL', 'source_url': 'https://www.nasdaq.com/articles/us-stocks-wall-st-edges-higher-as-powell-comments-bolster-peak-rate-bets', 'source_text': 'the media company and Apple AAPL.O have discussed bundling their streaming services at a discount.'}, {'triplet_id': '11', 'entity': "Wall Street's main indexes", 'entity_type': 'FIN_MARKET', 'relationship': 'Increases', 'target': 'Market Performance', 'target_type': 'FIN_METRIC', 'date': '2023-12-01 00:00:00+00:00', 'ticker': 'AAPL', 'source_url': 'https://www.nasdaq.com/articles/us-stocks-wall-st-edges-higher-as-powell-comments-bolster-peak-rate-bets', 'source_text': "Wall Street's main indexes inched higher on Friday"}, {'triplet_id': '12', 'entity': "Wall Street's main indexes", 'entity_type': 'FIN_MARKET', 'relationship': 'Market_Reacts_To', 'target': "Jerome Powell's comments", 'target_type': 'COMMENTARY', 'date': '2023-12-01 00:00:00+00:00', 'ticker': 'AAPL', 'source_url': 'https://www.nasdaq.com/articles/us-stocks-wall-st-edges-higher-as-powell-comments-bolster-peak-rate-bets', 'source_text': "Wall Street's main indexes inched higher on Friday after Federal Reserve Chair Jerome Powell acknowledged progress in lowering inflation"}, {'triplet_id': '13', 'entity': 'Jerome Powell', 'entity_type': 'PERSON', 'relationship': 'Announces', 'target': 'progress in lowering inflation', 'target_type': 'MACRO_CONDITION', 'date': '2023-12-01 00:00:00+00:00', 'ticker': 'AAPL', 'source_url': 'https://www.nasdaq.com/articles/us-stocks-wall-st-edges-higher-as-powell-comments-bolster-peak-rate-bets', 'source_text': 'Federal Reserve Chair Jerome Powell acknowledged progress in lowering inflation'}, {'triplet_id': '14', 'entity': 'Federal Reserve', 'entity_type': 'ORG_REG', 'relationship': 'Related_To', 'target': 'interest rate hiking campaign', 'target_type': 'MACRO_CONDITION', 'date': '2023-12-01 00:00:00+00:00', 'ticker': 'AAPL', 'source_url': 'https://www.nasdaq.com/articles/us-stocks-wall-st-edges-higher-as-powell-comments-bolster-peak-rate-bets', 'source_text': 'the central bank was done with its interest rate hiking campaign.'}, {'triplet_id': '15', 'entity': 'easing inflation', 'entity_type': 'MACRO_CONDITION', 'relationship': 'Positively_Impacts', 'target': 'equities', 'target_type': 'FIN_MARKET', 'date': '2023-12-01 00:00:00+00:00', 'ticker': 'AAPL', 'source_url': 'https://www.nasdaq.com/articles/us-stocks-wall-st-edges-higher-as-powell-comments-bolster-peak-rate-bets', 'source_text': 'signalled easing inflation and bolstered hopes the central bank would now end its interest rate hikes and could start lowering them soon, propelling a rally in equities.'}, {'triplet_id': '16', 'entity': 'S&P 500', 'entity_type': 'FIN_MARKET', 'relationship': 'Increases', 'target': 'Market Performance by nearly 9%', 'target_type': 'FIN_METRIC', 'date': '2023-12-01 00:00:00+00:00', 'ticker': 'AAPL', 'source_url': 'https://www.nasdaq.com/articles/wall-st-week-ahead-tax-loss-selling-santa-rally-could-sway-u.s.-stocks-after-november-melt', 'source_text': 'The S&P 500 gained nearly 9% gain in November'}, {'triplet_id': '17', 'entity': 'November', 'entity_type': 'EVENT', 'relationship': 'Positively_Impacts', 'target': 'S&P 500', 'target_type': 'FIN_MARKET', 'date': '2023-12-01 00:00:00+00:00', 'ticker': 'AAPL', 'source_url': 'https://www.nasdaq.com/articles/wall-st-week-ahead-tax-loss-selling-santa-rally-could-sway-u.s.-stocks-after-november-melt', 'source_text': 'November, historically a strong month for the index.'}, {'triplet_id': '18', 'entity': 'S&P 500', 'entity_type': 'FIN_MARKET', 'relationship': 'Increases', 'target': 'Market Performance 77% of the time', 'target_type': 'FIN_METRIC', 'date': '2023-12-01 00:00:00+00:00', 'ticker': 'AAPL', 'source_url': 'https://www.nasdaq.com/articles/wall-st-week-ahead-tax-loss-selling-santa-rally-could-sway-u.s.-stocks-after-november-melt', 'source_text': 'with the index rising 77% of the time'}, {'triplet_id': '19', 'entity': 'Tax-loss selling', 'entity_type': 'EVENT', 'relationship': 'Impacts', 'target': 'U.S. stocks', 'target_type': 'FIN_MARKET', 'date': '2023-12-01 00:00:00+00:00', 'ticker': 'AAPL', 'source_url': 'https://www.nasdaq.com/articles/wall-st-week-ahead-tax-loss-selling-santa-rally-could-sway-u.s.-stocks-after-november-melt', 'source_text': "Tax-loss selling, 'Santa rally' could sway U.S. stocks"}, {'triplet_id': '20', 'entity': "'Santa rally'", 'entity_type': 'EVENT', 'relationship': 'Impacts', 'target': 'U.S. stocks', 'target_type': 'FIN_MARKET', 'date': '2023-12-01 00:00:00+00:00', 'ticker': 'AAPL', 'source_url': 'https://www.nasdaq.com/articles/wall-st-week-ahead-tax-loss-selling-santa-rally-could-sway-u.s.-stocks-after-november-melt', 'source_text': "Tax-loss selling, 'Santa rally' could sway U.S. stocks"}, {'triplet_id': '21', 'entity': 'Apple', 'entity_type': 'ORG', 'relationship': 'Related_To', 'target': 'tech giants', 'target_type': 'SECTOR', 'date': '2023-12-01 00:00:00+00:00', 'ticker': 'AAPL', 'source_url': 'https://www.nasdaq.com/articles/2-tech-dividend-stocks-to-buy-and-hold-forever', 'source_text': 'Tech giants Apple (NASDAQ: AAPL)'}, {'triplet_id': '22', 'entity': 'Microsoft', 'entity_type': 'COMP', 'relationship': 'Related_To', 'target': 'tech giants', 'target_type': 'SECTOR', 'date': '2023-12-01 00:00:00+00:00', 'ticker': 'AAPL', 'source_url': 'https://www.nasdaq.com/articles/2-tech-dividend-stocks-to-buy-and-hold-forever', 'source_text': 'Microsoft (NASDAQ: MSFT)'}, {'triplet_id': '23', 'entity': 'Apple', 'entity_type': 'ORG', 'relationship': 'Increases', 'target': 'iPhone sales', 'target_type': 'FIN_METRIC', 'date': '2023-12-01 00:00:00+00:00', 'ticker': 'AAPL', 'source_url': 'https://www.nasdaq.com/articles/2-tech-dividend-stocks-to-buy-and-hold-forever', 'source_text': 'the company had its best fiscal Q4 ever for iPhone sales'}, {'triplet_id': '24', 'entity': "Apple's services unit", 'entity_type': 'SEGMENT', 'relationship': 'Increases', 'target': 'revenue', 'target_type': 'FIN_METRIC', 'date': '2023-12-01 00:00:00+00:00', 'ticker': 'AAPL', 'source_url': 'https://www.nasdaq.com/articles/2-tech-dividend-stocks-to-buy-and-hold-forever', 'source_text': "its services unit's revenue set an all-time high"}, {'triplet_id': '25', 'entity': 'Apple', 'entity_type': 'ORG', 'relationship': 'Member_Of', 'target': '"Magnificent Seven"', 'target_type': 'CONCEPT', 'date': '2023-12-01 00:00:00+00:00', 'ticker': 'AAPL', 'source_url': 'https://www.nasdaq.com/articles/graphic-resurgent-sp-500-crests-new-2023-closing-high-after-roller-coaster-year', 'source_text': 'The so-called "Magnificent Seven" -- Apple AAPL.O'}, {'triplet_id': '26', 'entity': 'Microsoft', 'entity_type': 'COMP', 'relationship': 'Member_Of', 'target': '"Magnificent Seven"', 'target_type': 'CONCEPT', 'date': '2023-12-01 00:00:00+00:00', 'ticker': 'AAPL', 'source_url': 'https://www.nasdaq.com/articles/graphic-resurgent-sp-500-crests-new-2023-closing-high-after-roller-coaster-year', 'source_text': 'Microsoft MSFT.O'}, {'triplet_id': '27', 'entity': 'Alphabet', 'entity_type': 'COMP', 'relationship': 'Member_Of', 'target': '"Magnificent Seven"', 'target_type': 'CONCEPT', 'date': '2023-12-01 00:00:00+00:00', 'ticker': 'AAPL', 'source_url': 'https://www.nasdaq.com/articles/graphic-resurgent-sp-500-crests-new-2023-closing-high-after-roller-coaster-year', 'source_text': 'Alphabet GOOGL.O'}, {'triplet_id': '28', 'entity': 'Amazon', 'entity_type': 'COMP', 'relationship': 'Member_Of', 'target': '"Magnificent Seven"', 'target_type': 'CONCEPT', 'date': '2023-12-01 00:00:00+00:00', 'ticker': 'AAPL', 'source_url': 'https://www.nasdaq.com/articles/graphic-resurgent-sp-500-crest-new-2023-closing-high-after-roller-coaster-year', 'source_text': 'Amazon AMZN.O'}, {'triplet_id': '29', 'entity': 'Nvidia', 'entity_type': 'COMP', 'relationship': 'Member_Of', 'target': '"Magnificent Seven"', 'target_type': 'CONCEPT', 'date': '2023-12-01 00:00:00+00:00', 'ticker': 'AAPL', 'source_url': 'https://www.nasdaq.com/articles/graphic-resurgent-sp-500-crests-new-2023-closing-high-after-roller-coaster-year', 'source_text': 'Nvidia NVDA.O'}, {'triplet_id': '30', 'entity': 'Meta Platforms', 'entity_type': 'COMP', 'relationship': 'Member_Of', 'target': '"Magnificent Seven"', 'target_type': 'CONCEPT', 'date': '2023-12-01 00:00:00+00:00', 'ticker': 'AAPL', 'source_url': 'https://www.nasdaq.com/articles/graphic-resurgent-sp-500-crests-new-2023-closing-high-after-roller-coaster-year', 'source_text': 'Meta Platforms META.O'}, {'triplet_id': '31', 'entity': 'Tesla', 'entity_type': 'COMP', 'relationship': 'Member_Of', 'target': '"Magnificent Seven"', 'target_type': 'CONCEPT', 'date': '2023-12-01 00:00:00+00:00', 'ticker': 'AAPL', 'source_url': 'https://www.nasdaq.com/articles/graphic-resurgent-sp-500-crests-new-2023-closing-high-after-roller-coaster-year', 'source_text': 'Tesla TSLA.O'}, {'triplet_id': '32', 'entity': '"Magnificent Seven"', 'entity_type': 'CONCEPT', 'relationship': 'Increases', 'target': 'stock gains of between about 47% and 220%', 'target_type': 'FIN_METRIC', 'date': '2023-12-01 00:00:00+00:00', 'ticker': 'AAPL', 'source_url': 'https://www.nasdaq.com/articles/graphic-resurgent-sp-500-crests-new-2023-closing-high-after-roller-coaster-year', 'source_text': 'The so-called "Magnificent Seven" -- Apple AAPL.O, Microsoft MSFT.O, Alphabet GOOGL.O, Amazon AMZN.O, Nvidia NVDA.O, Meta Platforms META.O and Tesla TSLA.O -- have seen stock gains of between about 47% and 220% so far this year.'}, {'triplet_id': '33', 'entity': 'Fed’s aggressive rate increases', 'entity_type': 'MACRO_CONDITION', 'relationship': 'Positively_Impacts', 'target': 'U.S. economy', 'target_type': 'ORG_GOV', 'date': '2023-12-01 00:00:00+00:00', 'ticker': 'AAPL', 'source_url': 'https://www.nasdaq.com/articles/graphic-resurgent-sp-500-crests-new-2023-closing-high-after-roller-coaster-year', 'source_text': 'the Fed’s aggressive rate increases so far appear to have done little damage to the U.S. economy'}, {'triplet_id': '34', 'entity': 'tighter monetary policy', 'entity_type': 'MACRO_CONDITION', 'relationship': 'Negatively_Impacts', 'target': 'growth', 'target_type': 'ECON_IND', 'date': '2023-12-01 00:00:00+00:00', 'ticker': 'AAPL', 'source_url': 'https://www.nasdaq.com/articles/graphic-resurgent-sp-500-crests-new-2023-closing-high-after-roller-coaster-year', 'source_text': 'fears that tighter monetary policy would hurt growth.'}, {'triplet_id': '35', 'entity': 'investors', 'entity_type': 'PERSON', 'relationship': 'Market_Reacts_To', 'target': 'Fed’s Nov. 1 meeting', 'target_type': 'EVENT', 'date': '2023-12-01 00:00:00+00:00', 'ticker': 'AAPL', 'source_url': 'https://www.nasdaq.com/articles/graphic-resurgent-sp-500-crests-new-2023-closing-high-after-roller-coaster-year', 'source_text': 'many investors came away from the Fed’s Nov. 1 meeting more confident that the central bank was close to wrapping up its rate increases.'}, {'triplet_id': '36', 'entity': 'Dow', 'entity_type': 'FIN_MARKET', 'relationship': 'Increases', 'target': 'Market Performance by 0.82%', 'target_type': 'FIN_METRIC', 'date': '2023-12-01 00:00:00+00:00', 'ticker': 'AAPL', 'source_url': 'https://www.nasdaq.com/articles/us-stocks-sp-500-hits-2023-closing-high-as-powell-strengthens-peak-rate-bets', 'source_text': 'Indexes up: Dow 0.82%'}, {'triplet_id': '37', 'entity': 'S&P', 'entity_type': 'FIN_MARKET', 'relationship': 'Increases', 'target': 'Market Performance by 0.59%', 'target_type': 'FIN_METRIC', 'date': '2023-12-01 00:00:00+00:00', 'ticker': 'AAPL', 'source_url': 'https://www.nasdaq.com/articles/us-stocks-sp-500-hits-2023-closing-high-as-powell-strengthens-peak-rate-bets', 'source_text': 'S&P 0.59%'}, {'triplet_id': '38', 'entity': 'Nasdaq', 'entity_type': 'FIN_MARKET', 'relationship': 'Increases', 'target': 'Market Performance by 0.55%', 'target_type': 'FIN_METRIC', 'date': '2023-12-01 00:00:00+00:00', 'ticker': 'AAPL', 'source_url': 'https://www.nasdaq.com/articles/us-stocks-sp-500-hits-2023-closing-high-as-powell-strengthens-peak-rate-bets', 'source_text': 'Nasdaq 0.55%'}, {'triplet_id': '39', 'entity': 'U.S. stocks', 'entity_type': 'FIN_MARKET', 'relationship': 'Increases', 'target': 'Market Performance', 'target_type': 'FIN_METRIC', 'date': '2023-12-01 00:00:00+00:00', 'ticker': 'AAPL', 'source_url': 'https://www.nasdaq.com/articles/us-stocks-sp-500-hits-2023-closing-high-as-powell-strengthens-peak-rate-bets', 'source_text': 'U.S. stocks rallied'}, {'triplet_id': '40', 'entity': 'S&P', 'entity_type': 'FIN_MARKET', 'relationship': 'Increases', 'target': 'Market Performance', 'target_type': 'FIN_METRIC', 'date': '2023-12-01 00:00:00+00:00', 'ticker': 'AAPL', 'source_url': 'https://www.nasdaq.com/articles/us-stocks-sp-500-hits-2023-closing-high-as-powell-strengthens-peak-rate-bets', 'source_text': 'the S&P registered its highest close of the year on Friday'}, {'triplet_id': '41', 'entity': "Jerome Powell's remarks", 'entity_type': 'COMMENTARY', 'relationship': 'Positively_Impacts', 'target': 'view that key policy rates have peaked', 'target_type': 'MACRO_CONDITION', 'date': '2023-12-01 00:00:00+00:00', 'ticker': 'AAPL', 'source_url': 'https://www.nasdaq.com/articles/us-stocks-sp-500-hits-2023-closing-high-as-powell-strengthens-peak-rate-bets', 'source_text': 'remarks from Federal Reserve Chair Jerome Powell bolstered the view that key policy rates have peaked.'}, {'triplet_id': '42', 'entity': 'S&P 500', 'entity_type': 'FIN_MARKET', 'relationship': 'Increases', 'target': 'one-month percentage gains', 'target_type': 'FIN_METRIC', 'date': '2023-12-01 00:00:00+00:00', 'ticker': 'AAPL', 'source_url': 'https://www.nasdaq.com/articles/us-stocks-sp-500-hits-2023-closing-high-as-powell-strengthens-peak-rate-bets', 'source_text': 'the S&P 500 and the Nasdaq registered their biggest one-month percentage gains since July 2022'}, {'triplet_id': '43', 'entity': 'Nasdaq', 'entity_type': 'FIN_MARKET', 'relationship': 'Increases', 'target': 'one-month percentage gains', 'target_type': 'FIN_METRIC', 'date': '2023-12-01 00:00:00+00:00', 'ticker': 'AAPL', 'source_url': 'https://www.nasdaq.com/articles/us-stocks-sp-500-hits-2023-closing-high-as-powell-strengthens-peak-rate-bets', 'source_text': 'the S&P 500 and the Nasdaq registered their biggest one-month percentage gains since July 2022'}, {'triplet_id': '44', 'entity': 'Dow', 'entity_type': 'FIN_MARKET', 'relationship': 'Increases', 'target': 'Market Performance', 'target_type': 'FIN_METRIC', 'date': '2023-12-01 00:00:00+00:00', 'ticker': 'AAPL', 'source_url': 'https://www.nasdaq.com/articles/us-stocks-sp-500-hits-2023-closing-high-as-powell-strengthens-peak-rate-bets', 'source_text': 'the Dow closed at its highest level since January 2022.'}, {'triplet_id': '45', 'entity': 'Paramount Global', 'entity_type': 'COMP', 'relationship': 'Increases', 'target': 'Share Price by 9.6%', 'target_type': 'FIN_METRIC', 'date': '2023-12-01 00:00:00+00:00', 'ticker': 'AAPL', 'source_url': 'https://www.nasdaq.com/articles/us-stocks-wall-st-rallies-as-powell-cements-peak-rate-bets', 'source_text': 'Paramount GlobalPARA.O jumped 9.6%'}, {'triplet_id': '46', 'entity': 'Paramount Global', 'entity_type': 'COMP', 'relationship': 'Partners_With', 'target': 'Apple', 'target_type': 'ORG', 'date': '2023-12-01 00:00:00+00:00', 'ticker': 'AAPL', 'source_url': 'https://www.nasdaq.com/articles/us-stocks-wall-st-rallies-as-powell-cements-peak-rate-bets', 'source_text': 'the media company and Apple AAPL.O have discussed bundling their streaming services at a discount.'}, {'triplet_id': '47', 'entity': 'Apple', 'entity_type': 'ORG', 'relationship': 'Partners_With', 'target': 'Paramount Global', 'target_type': 'COMP', 'date': '2023-12-01 00:00:00+00:00', 'ticker': 'AAPL', 'source_url': 'https://www.nasdaq.com/articles/us-stocks-wall-st-rallies-as-powell-cements-peak-rate-bets', 'source_text': 'the media company and Apple AAPL.O have discussed bundling their streaming services at a discount.'}, {'triplet_id': '48', 'entity': 'U.S. stocks', 'entity_type': 'FIN_MARKET', 'relationship': 'Increases', 'target': 'Market Performance', 'target_type': 'FIN_METRIC', 'date': '2023-12-01 00:00:00+00:00', 'ticker': 'AAPL', 'source_url': 'https://www.nasdaq.com/articles/us-stocks-wall-st-rallies-as-powell-cements-peak-rate-bets', 'source_text': 'U.S. stocks advanced on Friday'}, {'triplet_id': '49', 'entity': "Jerome Powell's remarks", 'entity_type': 'COMMENTARY', 'relationship': 'Positively_Impacts', 'target': 'view that interest rates have peaked', 'target_type': 'MACRO_CONDITION', 'date': '2023-12-01 00:00:00+00:00', 'ticker': 'AAPL', 'source_url': 'https://www.nasdaq.com/articles/us-stocks-wall-st-rallies-as-powell-cements-peak-rate-bets', 'source_text': 'remarks from Federal Reserve Chairman Jerome Powell bolstered the view that interest rates have peaked.'}, {'triplet_id': '50', 'entity': 'All three major U.S. stock indexes', 'entity_type': 'FIN_MARKET', 'relationship': 'Increases', 'target': 'Market Performance', 'target_type': 'FIN_METRIC', 'date': '2023-12-01 00:00:00+00:00', 'ticker': 'AAPL', 'source_url': 'https://www.nasdaq.com/articles/us-stocks-wall-st-rallies-as-powell-cements-peak-rate-bets', 'source_text': 'All three major U.S. stock indexes were higher'}, {'triplet_id': '51', 'entity': 'Digital Markets Act (DMA)', 'entity_type': 'REGULATORY_REQUIREMENT', 'relationship': 'Regulates', 'target': 'TikTok', 'target_type': 'COMP', 'date': '2023-12-01 00:00:00+00:00', 'ticker': 'AAPL', 'source_url': 'https://www.nasdaq.com/articles/tiktok-asks-eu-court-to-suspend-eu-gatekeeper-label-until-its-ruling', 'source_text': "The Digital Markets Act (DMA) requires TikTok and other designated gatekeepers Alphabet's GOOGL.O Google, Meta Platforms META.O, Apple AAPL.O, Amazon AMZN.O and Microsoft MSFT.O to make their messaging apps interoperate with rivals"}, {'triplet_id': '52', 'entity': 'Digital Markets Act (DMA)', 'entity_type': 'REGULATORY_REQUIREMENT', 'relationship': 'Regulates', 'target': "Alphabet's Google", 'target_type': 'COMP', 'date': '2023-12-01 00:00:00+00:00', 'ticker': 'AAPL', 'source_url': 'https://www.nasdaq.com/articles/tiktok-asks-eu-court-to-suspend-eu-gatekeeper-label-until-its-ruling', 'source_text': "The Digital Markets Act (DMA) requires TikTok and other designated gatekeepers Alphabet's GOOGL.O Google, Meta Platforms META.O, Apple AAPL.O, Amazon AMZN.O and Microsoft MSFT.O to make their messaging apps interoperate with rivals"}, {'triplet_id': '53', 'entity': 'Digital Markets Act (DMA)', 'entity_type': 'REGULATORY_REQUIREMENT', 'relationship': 'Regulates', 'target': 'Meta Platforms', 'target_type': 'COMP', 'date': '2023-12-01 00:00:00+00:00', 'ticker': 'AAPL', 'source_url': 'https://www.nasdaq.com/articles/tiktok-asks-eu-court-to-suspend-eu-gatekeeper-label-until-its-ruling', 'source_text': "The Digital Markets Act (DMA) requires TikTok and other designated gatekeepers Alphabet's GOOGL.O Google, Meta Platforms META.O, Apple AAPL.O, Amazon AMZN.O and Microsoft MSFT.O to make their messaging apps interoperate with rivals"}, {'triplet_id': '54', 'entity': 'Digital Markets Act (DMA)', 'entity_type': 'REGULATORY_REQUIREMENT', 'relationship': 'Regulates', 'target': 'Apple', 'target_type': 'ORG', 'date': '2023-12-01 00:00:00+00:00', 'ticker': 'AAPL', 'source_url': 'https://www.nasdaq.com/articles/tiktok-asks-eu-court-to-suspend-eu-gatekeeper-label-until-its-ruling', 'source_text': "The Digital Markets Act (DMA) requires TikTok and other designated gatekeepers Alphabet's GOOGL.O Google, Meta Platforms META.O, Apple AAPL.O, Amazon AMZN.O and Microsoft MSFT.O to make their messaging apps interoperate with rivals"}, {'triplet_id': '55', 'entity': 'Digital Markets Act (DMA)', 'entity_type': 'REGULATORY_REQUIREMENT', 'relationship': 'Regulates', 'target': 'Amazon', 'target_type': 'COMP', 'date': '2023-12-01 00:00:00+00:00', 'ticker': 'AAPL', 'source_url': 'https://www.nasdaq.com/articles/tiktok-asks-eu-court-to-suspend-eu-gatekeeper-label-until-its-ruling', 'source_text': "The Digital Markets Act (DMA) requires TikTok and other designated gatekeepers Alphabet's GOOGL.O Google, Meta Platforms META.O, Apple AAPL.O, Amazon AMZN.O and Microsoft MSFT.O to make their messaging apps interoperate with rivals"}, {'triplet_id': '56', 'entity': 'Digital Markets Act (DMA)', 'entity_type': 'REGULATORY_REQUIREMENT', 'relationship': 'Regulates', 'target': 'Microsoft', 'target_type': 'COMP', 'date': '2023-12-01 00:00:00+00:00', 'ticker': 'AAPL', 'source_url': 'https://www.nasdaq.com/articles/tiktok-asks-eu-court-to-suspend-eu-gatekeeper-label-until-its-ruling', 'source_text': "The Digital Markets Act (DMA) requires TikTok and other designated gatekeepers Alphabet's GOOGL.O Google, Meta Platforms META.O, Apple AAPL.O, Amazon AMZN.O and Microsoft MSFT.O to make their messaging apps interoperate with rivals"}, {'triplet_id': '57', 'entity': 'TikTok', 'entity_type': 'COMP', 'relationship': 'Involved_In', 'target': 'suspend its designation as a gatekeeper', 'target_type': 'LITIGATION', 'date': '2023-12-01 00:00:00+00:00', 'ticker': 'AAPL', 'source_url': 'https://www.nasdaq.com/articles/tiktok-asks-eu-court-to-suspend-eu-gatekeeper-label-until-its-ruling', 'source_text': "TikTok has asked Europe's second highest court to suspend its designation as a gatekeeper under onerous new EU tech rules"}, {'triplet_id': '58', 'entity': 'TikTok', 'entity_type': 'COMP', 'relationship': 'Subject_To', 'target': 'new EU tech rules', 'target_type': 'REGULATORY_REQUIREMENT', 'date': '2023-12-01 00:00:00+00:00', 'ticker': 'AAPL', 'source_url': 'https://www.nasdaq.com/articles/tiktok-asks-eu-court-to-suspend-eu-gatekeeper-label-until-its-ruling', 'source_text': 'suspend its designation as a gatekeeper under onerous new EU tech rules'}, {'triplet_id': '59', 'entity': "ByteDance's TikTok", 'entity_type': 'COMP', 'relationship': 'Member_Of', 'target': 'Chinese conglomerate ByteDance', 'target_type': 'COMP', 'date': '2023-12-01 00:00:00+00:00', 'ticker': 'AAPL', 'source_url': 'https://www.nasdaq.com/articles/tiktok-asks-eu-court-to-suspend-eu-gatekeeper-label-until-its-ruling', 'source_text': "Chinese conglomerate ByteDance's TikTok"}, {'triplet_id': '60', 'entity': 'Walmart', 'entity_type': 'COMP', 'relationship': 'Negatively_Impacts', 'target': 'social platform X', 'target_type': 'PRODUCT', 'date': '2023-12-01 00:00:00+00:00', 'ticker': 'AAPL', 'source_url': 'https://www.nasdaq.com/articles/walmart-says-it-is-not-advertising-on-social-platform-x-0', 'source_text': 'Walmart WMT.N said on Friday it is not advertising on social media platform X'}, {'triplet_id': '61', 'entity': 'Apple', 'entity_type': 'ORG', 'relationship': 'Negatively_Impacts', 'target': 'X', 'target_type': 'PRODUCT', 'date': '2023-12-01 00:00:00+00:00', 'ticker': 'AAPL', 'source_url': 'https://www.nasdaq.com/articles/walmart-says-it-is-not-advertising-on-social-platform-x-0', 'source_text': 'Apple AAPL.O, Walt Disney DIS.N and Warner Bros Discovery WBD.O also suspended their ads on X this month'}, {'triplet_id': '62', 'entity': 'Walt Disney', 'entity_type': 'COMP', 'relationship': 'Negatively_Impacts', 'target': 'X', 'target_type': 'PRODUCT', 'date': '2023-12-01 00:00:00+00:00', 'ticker': 'AAPL', 'source_url': 'https://www.nasdaq.com/articles/walmart-says-it-is-not-advertising-on-social-platform-x-0', 'source_text': 'Walt Disney DIS.N and Warner Bros Discovery WBD.O also suspended their ads on X this month'}, {'triplet_id': '63', 'entity': 'Warner Bros Discovery', 'entity_type': 'COMP', 'relationship': 'Negatively_Impacts', 'target': 'X', 'target_type': 'PRODUCT', 'date': '2023-12-01 00:00:00+00:00', 'ticker': 'AAPL', 'source_url': 'https://www.nasdaq.com/articles/walmart-says-it-is-not-advertising-on-social-platform-x-0', 'source_text': 'Warner Bros Discovery WBD.O also suspended their ads on X this month'}, {'triplet_id': '64', 'entity': 'antisemitic posts', 'entity_type': 'RISK_FACTOR', 'relationship': 'Negatively_Impacts', 'target': 'ads on X', 'target_type': 'PRODUCT', 'date': '2023-12-01 00:00:00+00:00', 'ticker': 'AAPL', 'source_url': 'https://www.nasdaq.com/articles/walmart-says-it-is-not-advertising-on-social-platform-x-0', 'source_text': 'ads had appeared next to antisemitic posts.'}, {'triplet_id': '65', 'entity': 'X', 'entity_type': 'PRODUCT', 'relationship': 'Has_Stake_In', 'target': 'Elon Musk', 'target_type': 'PERSON', 'date': '2023-12-01 00:00:00+00:00', 'ticker': 'AAPL', 'source_url': 'https://www.nasdaq.com/articles/walmart-says-it-is-not-advertising-on-social-platform-x-0', 'source_text': 'Elon Musk-owned site.'}, {'triplet_id': '66', 'entity': 'Alphabet', 'entity_type': 'COMP', 'relationship': 'Related_To', 'target': 'Amazon', 'target_type': 'COMP', 'date': '2023-12-01 00:00:00+00:00', 'ticker': 'AAPL', 'source_url': 'https://www.nasdaq.com/articles/alphabet-googl-bolsters-nest-hub-series-with-fuchsia-14', 'source_text': 'Alphabet to compete well with some notable industry players like Amazon AMZN and Apple AAPL, which are also making concerted efforts to gain a solid footing in the smart home market space.'}, {'triplet_id': '67', 'entity': 'Alphabet', 'entity_type': 'COMP', 'relationship': 'Related_To', 'target': 'Apple', 'target_type': 'ORG', 'date': '2023-12-01 00:00:00+00:00', 'ticker': 'AAPL', 'source_url': 'https://www.nasdaq.com/articles/alphabet-googl-bolsters-nest-hub-series-with-fuchsia-14', 'source_text': 'Alphabet to compete well with some notable industry players like Amazon AMZN and Apple AAPL, which are also making concerted efforts to gain a solid footing in the smart home market space.'}, {'triplet_id': '68', 'entity': 'Amazon', 'entity_type': 'COMP', 'relationship': 'Invests_In', 'target': 'smart home market space', 'target_type': 'SECTOR', 'date': '2023-12-01 00:00:00+00:00', 'ticker': 'AAPL', 'source_url': 'https://www.nasdaq.com/articles/alphabet-googl-bolsters-nest-hub-series-with-fuchsia-14', 'source_text': 'Amazon AMZN and Apple AAPL, which are also making concerted efforts to gain a solid footing in the smart home market space.'}, {'triplet_id': '69', 'entity': 'Apple', 'entity_type': 'ORG', 'relationship': 'Invests_In', 'target': 'smart home market space', 'target_type': 'SECTOR', 'date': '2023-12-01 00:00:00+00:00', 'ticker': 'AAPL', 'source_url': 'https://www.nasdaq.com/articles/alphabet-googl-bolsters-nest-hub-series-with-fuchsia-14', 'source_text': 'Amazon AMZN and Apple AAPL, which are also making concerted efforts to gain a solid footing in the smart home market space.'}, {'triplet_id': '70', 'entity': 'Alphabet', 'entity_type': 'COMP', 'relationship': 'Introduces', 'target': 'Google smart home devices portfolio', 'target_type': 'PRODUCT', 'date': '2023-12-01 00:00:00+00:00', 'ticker': 'AAPL', 'source_url': 'https://www.nasdaq.com/articles/alphabet-googl-bolsters-nest-hub-series-with-fuchsia-14', 'source_text': 'Alphabet GOOGL is enhancing its Google smart home devices portfolio on the back of new feature updates and releases.'}, {'triplet_id': '71', 'entity': 'AAPL stock', 'entity_type': 'FIN_INST', 'relationship': 'Related_To', 'target': 'closing price of $189.95', 'target_type': 'FIN_METRIC', 'date': '2023-12-01 00:00:00+00:00', 'ticker': 'AAPL', 'source_url': 'https://www.nasdaq.com/articles/looking-for-income-these-3-unusually-active-options-should-generate-income-over-the-next-7', 'source_text': 'closing price of $189.95, AAPL stock'}, {'triplet_id': '72', 'entity': 'AAPL stock', 'entity_type': 'FIN_INST', 'relationship': 'Increases', 'target': 'Share Price by 2.6%', 'target_type': 'FIN_METRIC', 'date': '2023-12-01 00:00:00+00:00', 'ticker': 'AAPL', 'source_url': 'https://www.nasdaq.com/articles/looking-for-income-these-3-unusually-active-options-should-generate-income-over-the-next-7', 'source_text': 'AAPL stock has to rise by 2.6%'}, {'triplet_id': '73', 'entity': 'Apple', 'entity_type': 'ORG', 'relationship': 'Related_To', 'target': 'seven unusually active options', 'target_type': 'FIN_INST', 'date': '2023-12-01 00:00:00+00:00', 'ticker': 'AAPL', 'source_url': 'https://www.nasdaq.com/articles/looking-for-income-these-3-unusually-active-options-should-generate-income-over-the-next-7', 'source_text': 'Apple (AAPL) had seven unusually active options on Thursday'}, {'triplet_id': '74', 'entity': 'AAPL stock', 'entity_type': 'FIN_INST', 'relationship': 'Increases', 'target': 'Share Price by $1.40', 'target_type': 'FIN_METRIC', 'date': '2023-12-01 00:00:00+00:00', 'ticker': 'AAPL', 'source_url': 'https://www.nasdaq.com/articles/looking-for-income-these-3-unusually-active-options-should-generate-income-over-the-next-7', 'source_text': 'AAPL stock is up $1.40 in Friday trading'}, {'triplet_id': '75', 'entity': 'Paramount Global', 'entity_type': 'COMP', 'relationship': 'Increases', 'target': 'Share Price by 1%', 'target_type': 'FIN_METRIC', 'date': '2023-12-01 00:00:00+00:00', 'ticker': 'AAPL', 'source_url': 'https://www.nasdaq.com/articles/us-stocks-sp-nasdaq-slip-as-caution-prevails-ahead-of-powell-comments', 'source_text': 'Paramount GlobalPARA.O climbed 1%'}, {'triplet_id': '76', 'entity': 'Paramount Global', 'entity_type': 'COMP', 'relationship': 'Partners_With', 'target': 'Apple', 'target_type': 'ORG', 'date': '2023-12-01 00:00:00+00:00', 'ticker': 'AAPL', 'source_url': 'https://www.nasdaq.com/articles/us-stocks-sp-nasdaq-slip-as-caution-prevails-ahead-of-powell-comments', 'source_text': 'the media company and Apple AAPL.O have discussed bundling their streaming services at a discount.'}, {'triplet_id': '77', 'entity': 'Apple', 'entity_type': 'ORG', 'relationship': 'Partners_With', 'target': 'Paramount Global', 'target_type': 'COMP', 'date': '2023-12-01 00:00:00+00:00', 'ticker': 'AAPL', 'source_url': 'https://www.nasdaq.com/articles/us-stocks-sp-nasdaq-slip-as-caution-prevails-ahead-of-powell-comments', 'source_text': 'the media company and Apple AAPL.O have discussed bundling their streaming services at a discount.'}, {'triplet_id': '78', 'entity': 'S&P 500', 'entity_type': 'FIN_MARKET', 'relationship': 'Decreases', 'target': 'Market Performance', 'target_type': 'FIN_METRIC', 'date': '2023-12-01 00:00:00+00:00', 'ticker': 'AAPL', 'source_url': 'https://www.nasdaq.com/articles/us-stocks-sp-nasdaq-slip-as-caution-prevails-ahead-of-powell-comments', 'source_text': 'The S&P 500 and Nasdaq fell on Friday'}, {'triplet_id': '79', 'entity': 'Nasdaq', 'entity_type': 'FIN_MARKET', 'relationship': 'Decreases', 'target': 'Market Performance', 'target_type': 'FIN_METRIC', 'date': '2023-12-01 00:00:00+00:00', 'ticker': 'AAPL', 'source_url': 'https://www.nasdaq.com/articles/us-stocks-sp-nasdaq-slip-as-caution-prevails-ahead-of-powell-comments', 'source_text': 'The S&P 500 and Nasdaq fell on Friday'}, {'triplet_id': '80', 'entity': 'S&P 500 and Nasdaq', 'entity_type': 'FIN_MARKET', 'relationship': 'Market_Reacts_To', 'target': "Powell's comments", 'target_type': 'COMMENTARY', 'date': '2023-12-01 00:00:00+00:00', 'ticker': 'AAPL', 'source_url': 'https://www.nasdaq.com/articles/us-stocks-sp-nasdaq-slip-as-caution-prevails-ahead-of-powell-comments', 'source_text': "The S&P 500 and Nasdaq fell on Friday as investors were on edge ahead of Federal Reserve Chair Jerome Powell's comments"}, {'triplet_id': '81', 'entity': 'S&P 500', 'entity_type': 'FIN_MARKET', 'relationship': 'Increases', 'target': 'monthly gain', 'target_type': 'FIN_METRIC', 'date': '2023-12-01 00:00:00+00:00', 'ticker': 'AAPL', 'source_url': 'https://www.nasdaq.com/articles/us-stocks-sp-nasdaq-slip-as-caution-prevails-ahead-of-powell-comments', 'source_text': 'both the indexes finished November with their biggest monthly gain since July 2022'}, {'triplet_id': '82', 'entity': 'Nasdaq', 'entity_type': 'FIN_MARKET', 'relationship': 'Increases', 'target': 'monthly gain', 'target_type': 'FIN_METRIC', 'date': '2023-12-01 00:00:00+00:00', 'ticker': 'AAPL', 'source_url': 'https://www.nasdaq.com/articles/us-stocks-sp-nasdaq-slip-as-caution-prevails-ahead-of-powell-comments', 'source_text': 'both the indexes finished November with their biggest monthly gain since July 2022'}, {'triplet_id': '83', 'entity': 'Dow Jones', 'entity_type': 'FIN_MARKET', 'relationship': 'Increases', 'target': 'Market Performance', 'target_type': 'FIN_METRIC', 'date': '2023-12-01 00:00:00+00:00', 'ticker': 'AAPL', 'source_url': 'https://www.nasdaq.com/articles/us-stocks-sp-nasdaq-slip-as-caution-prevails-ahead-of-powell-comments', 'source_text': 'the Dow Jones .DJI rallied to close at its highest level since January 2022.'}, {'triplet_id': '84', 'entity': 'Apple', 'entity_type': 'ORG', 'relationship': 'Has_Stake_In', 'target': '36% of the Google Search revenue', 'target_type': 'FIN_METRIC', 'date': '2023-12-01 00:00:00+00:00', 'ticker': 'AAPL', 'source_url': 'https://www.nasdaq.com/articles/1-glaring-risk-for-apple-stock-investors-that-just-got-more-concerning', 'source_text': 'Apple (NASDAQ: AAPL) receives 36% of the Google Search revenue that goes through the Safari browser'}, {'triplet_id': '85', 'entity': 'Apple', 'entity_type': 'ORG', 'relationship': 'Introduces', 'target': 'Safari browser', 'target_type': 'PRODUCT', 'date': '2023-12-01 00:00:00+00:00', 'ticker': 'AAPL', 'source_url': 'https://www.nasdaq.com/articles/1-glaring-risk-for-apple-stock-investors-that-just-got-more-concerning', 'source_text': 'Safari browser that Apple pre-installs on its devices.'}, {'triplet_id': '86', 'entity': 'courts rule against Alphabet', 'entity_type': 'LITIGATION', 'relationship': 'Negatively_Impacts', 'target': 'Apple', 'target_type': 'ORG', 'date': '2023-12-01 00:00:00+00:00', 'ticker': 'AAPL', 'source_url': 'https://www.nasdaq.com/articles/1-glaring-risk-for-apple-stock-investors-that-just-got-more-concerning', 'source_text': 'If the courts rule against Alphabet, it is unlikely that Apple would be able to replicate its deal with any other search engine provider.'}, {'triplet_id': '87', 'entity': 'Google Search', 'entity_type': 'PRODUCT', 'relationship': 'Related_To', 'target': '$55 billion in revenue', 'target_type': 'FIN_METRIC', 'date': '2023-12-01 00:00:00+00:00', 'ticker': 'AAPL', 'source_url': 'https://www.nasdaq.com/articles/1-glaring-risk-for-apple-stock-investors-that-just-got-more-concerning', 'source_text': 'Google Search earns about $55 billion in revenue through Safari.'}, {'triplet_id': '88', 'entity': 'AAPL', 'entity_type': 'ORG', 'relationship': 'Related_To', 'target': 'Twin Momentum Investor model', 'target_type': 'CONCEPT', 'date': '2023-12-01 00:00:00+00:00', 'ticker': 'AAPL', 'source_url': 'https://www.nasdaq.com/articles/validea-detailed-fundamental-analysis-aapl-10', 'source_text': 'AAPL rates highest using our Twin Momentum Investor model'}, {'triplet_id': '89', 'entity': 'Twin Momentum Investor model', 'entity_type': 'CONCEPT', 'relationship': 'Related_To', 'target': "Dashan Huang's strategy", 'target_type': 'PERSON', 'date': '2023-12-01 00:00:00+00:00', 'ticker': 'AAPL', 'source_url': 'https://www.nasdaq.com/articles/validea-detailed-fundamental-analysis-aapl-10', 'source_text': 'based on the published strategy of Dashan Huang.'}, {'triplet_id': '90', 'entity': 'APPLE INC', 'entity_type': 'ORG', 'relationship': 'Related_To', 'target': 'large-cap growth stock', 'target_type': 'FIN_INST', 'date': '2023-12-01 00:00:00+00:00', 'ticker': 'AAPL', 'source_url': 'https://www.nasdaq.com/articles/validea-detailed-fundamental-analysis-aapl-10', 'source_text': 'APPLE INC (AAPL) is a large-cap growth stock'}, {'triplet_id': '91', 'entity': 'APPLE INC', 'entity_type': 'ORG', 'relationship': 'Operates_In', 'target': 'Communications Equipment industry', 'target_type': 'SECTOR', 'date': '2023-12-01 00:00:00+00:00', 'ticker': 'AAPL', 'source_url': 'https://www.nasdaq.com/articles/validea-detailed-fundamental-analysis-aapl-10', 'source_text': 'in the Communications Equipment industry.'}, {'triplet_id': '92', 'entity': 'Apple Inc.', 'entity_type': 'ORG', 'relationship': 'Partners_With', 'target': 'Paramount Global', 'target_type': 'COMP', 'date': '2023-12-01 00:00:00+00:00', 'ticker': 'AAPL', 'source_url': 'https://www.nasdaq.com/articles/wsj%3A-apple-in-talks-to-bundle-paramount-with-apple-tv', 'source_text': 'Apple Inc. (AAPL) and Paramount Global are in talks to bundle Paramount+ and Apple TV+ together'}, {'triplet_id': '93', 'entity': 'Paramount Global', 'entity_type': 'COMP', 'relationship': 'Partners_With', 'target': 'Apple Inc.', 'target_type': 'ORG', 'date': '2023-12-01 00:00:00+00:00', 'ticker': 'AAPL', 'source_url': 'https://www.nasdaq.com/articles/wsj%3A-apple-in-talks-to-bundle-paramount-with-apple-tv', 'source_text': 'Apple Inc. (AAPL) and Paramount Global are in talks to bundle Paramount+ and Apple TV+ together'}, {'triplet_id': '94', 'entity': 'Paramount+', 'entity_type': 'PRODUCT', 'relationship': 'Related_To', 'target': 'direct-to-consumer digital subscription video on-demand and live streaming service', 'target_type': 'PRODUCT', 'date': '2023-12-01 00:00:00+00:00', 'ticker': 'AAPL', 'source_url': 'https://www.nasdaq.com/articles/wsj%3A-apple-in-talks-to-bundle-paramount-with-apple-tv', 'source_text': 'Paramount+ is a direct-to-consumer digital subscription video on-demand and live streaming service'}, {'triplet_id': '95', 'entity': 'Paramount', 'entity_type': 'COMP', 'relationship': 'Produces', 'target': 'premium content', 'target_type': 'PRODUCT', 'date': '2023-12-01 00:00:00+00:00', 'ticker': 'AAPL', 'source_url': 'https://www.nasdaq.com/articles/wsj%3A-apple-in-talks-to-bundle-paramount-with-apple-tv', 'source_text': 'Paramount delivers premium content to audiences across platforms worldwide.'}, {'triplet_id': '96', 'entity': 'Apple', 'entity_type': 'ORG', 'relationship': 'Partners_With', 'target': 'Paramount Global', 'target_type': 'COMP', 'date': '2023-12-01 00:00:00+00:00', 'ticker': 'AAPL', 'source_url': 'https://www.nasdaq.com/articles/apple-paramount-discuss-bundling-their-streaming-services-wsj', 'source_text': 'Apple AAPL.O and Paramount Global PARA.O have discussed bundling their streaming services at a discount'}, {'triplet_id': '97', 'entity': 'Paramount Global', 'entity_type': 'COMP', 'relationship': 'Partners_With', 'target': 'Apple', 'target_type': 'ORG', 'date': '2023-12-01 00:00:00+00:00', 'ticker': 'AAPL', 'source_url': 'https://www.nasdaq.com/articles/apple-paramount-discuss-bundling-their-streaming-services-wsj', 'source_text': 'Apple AAPL.O and Paramount Global PARA.O have discussed bundling their streaming services at a discount'}, {'triplet_id': '98', 'entity': 'Shares of Paramount', 'entity_type': 'FIN_INST', 'relationship': 'Increases', 'target': 'Share Price by 3%', 'target_type': 'FIN_METRIC', 'date': '2023-12-01 00:00:00+00:00', 'ticker': 'AAPL', 'source_url': 'https://www.nasdaq.com/articles/apple-paramount-discuss-bundling-their-streaming-services-wsj', 'source_text': 'Shares of media company Paramount rose 3% in premarket trading.'}, {'triplet_id': '99', 'entity': 'Nvidia', 'entity_type': 'COMP', 'relationship': 'Related_To', 'target': 'Apple', 'target_type': 'ORG', 'date': '2023-12-01 00:00:00+00:00', 'ticker': 'AAPL', 'source_url': 'https://www.nasdaq.com/articles/nvidia-remains-a-must-own-growth-stock.-heres-why.', 'source_text': 'Nvidia a cheaper stock to buy than either Apple (NASDAQ:AAPL)'}, {'triplet_id': '100', 'entity': 'Nvidia', 'entity_type': 'COMP', 'relationship': 'Related_To', 'target': 'Microsoft', 'target_type': 'COMP', 'date': '2023-12-01 00:00:00+00:00', 'ticker': 'AAPL', 'source_url': 'https://www.nasdaq.com/articles/nvidia-remains-a-must-own-growth-stock.-heres-why.', 'source_text': 'Nvidia a cheaper stock to buy than either Apple (NASDAQ:AAPL) or Microsoft.'}, {'triplet_id': '101', 'entity': 'Joel Baglole', 'entity_type': 'PERSON', 'relationship': 'Has_Stake_In', 'target': 'NVDA', 'target_type': 'FIN_INST', 'date': '2023-12-01 00:00:00+00:00', 'ticker': 'AAPL', 'source_url': 'https://www.nasdaq.com/articles/nvidia-remains-a-must-own-growth-stock.-heres-why.', 'source_text': 'Joel Baglole held long positions in NVDA'}, {'triplet_id': '102', 'entity': 'Joel Baglole', 'entity_type': 'PERSON', 'relationship': 'Has_Stake_In', 'target': 'MSFT', 'target_type': 'FIN_INST', 'date': '2023-12-01 00:00:00+00:00', 'ticker': 'AAPL', 'source_url': 'https://www.nasdaq.com/articles/nvidia-remains-a-must-own-growth-stock.-heres-why.', 'source_text': 'Joel Baglole held long positions in NVDA, MSFT'}, {'triplet_id': '103', 'entity': 'Joel Baglole', 'entity_type': 'PERSON', 'relationship': 'Has_Stake_In', 'target': 'AAPL', 'target_type': 'FIN_INST', 'date': '2023-12-01 00:00:00+00:00', 'ticker': 'AAPL', 'source_url': 'https://www.nasdaq.com/articles/nvidia-remains-a-must-own-growth-stock.-heres-why.', 'source_text': 'Joel Baglole held long positions in NVDA, MSFT and AAPL.'}, {'triplet_id': '104', 'entity': 'global demand', 'entity_type': 'MACRO_CONDITION', 'relationship': 'Increases', 'target': "Nvidia's microchips", 'target_type': 'PRODUCT', 'date': '2023-12-01 00:00:00+00:00', 'ticker': 'AAPL', 'source_url': 'https://www.nasdaq.com/articles/nvidia-remains-a-must-own-growth-stock.-heres-why.', 'source_text': 'global demand for its microchips and semiconductors accelerating'}, {'triplet_id': '105', 'entity': 'global demand', 'entity_type': 'MACRO_CONDITION', 'relationship': 'Increases', 'target': "Nvidia's semiconductors", 'target_type': 'PRODUCT', 'date': '2023-12-01 00:00:00+00:00', 'ticker': 'AAPL', 'source_url': 'https://www.nasdaq.com/articles/nvidia-remains-a-must-own-growth-stock.-heres-why.', 'source_text': 'global demand for its microchips and semiconductors accelerating'}, {'triplet_id': '106', 'entity': 'development of AI', 'entity_type': 'CONCEPT', 'relationship': 'Related_To', 'target': 'infancy', 'target_type': 'CONCEPT', 'date': '2023-12-01 00:00:00+00:00', 'ticker': 'AAPL', 'source_url': 'https://www.nasdaq.com/articles/nvidia-remains-a-must-own-growth-stock.-heres-why.', 'source_text': 'development of AI in its infancy'}, {'triplet_id': '107', 'entity': 'Nvidia stock', 'entity_type': 'FIN_INST', 'relationship': 'Related_To', 'target': 'growth play', 'target_type': 'CONCEPT', 'date': '2023-12-01 00:00:00+00:00', 'ticker': 'AAPL', 'source_url': 'https://www.nasdaq.com/articles/nvidia-remains-a-must-own-growth-stock.-heres-why.', 'source_text': 'Nvidia (NASDAQ:NVDA) stock remains a must-own growth play'}, {'triplet_id': '108', 'entity': 'The market', 'entity_type': 'FIN_MARKET', 'relationship': 'Increases', 'target': 'Market Performance', 'target_type': 'FIN_METRIC', 'date': '2023-12-01 00:00:00+00:00', 'ticker': 'AAPL', 'source_url': 'https://www.nasdaq.com/articles/should-you-really-invest-in-stocks-now-or-wait-until-the-new-year', 'source_text': 'The market is rallying'}, {'triplet_id': '109', 'entity': 'all three major indexes', 'entity_type': 'FIN_MARKET', 'relationship': 'Increases', 'target': 'Market Performance', 'target_type': 'FIN_METRIC', 'date': '2023-12-01 00:00:00+00:00', 'ticker': 'AAPL', 'source_url': 'https://www.nasdaq.com/articles/should-you-really-invest-in-stocks-now-or-wait-until-the-new-year', 'source_text': 'all three major indexes climbing'}, {'triplet_id': '110', 'entity': 'Amazon', 'entity_type': 'COMP', 'relationship': 'Positively_Impacts', 'target': 'top stocks', 'target_type': 'FIN_INST', 'date': '2023-12-01 00:00:00+00:00', 'ticker': 'AAPL', 'source_url': 'https://www.nasdaq.com/articles/should-you-really-invest-in-stocks-now-or-wait-until-the-new-year', 'source_text': 'top stocks such as Amazon (NASDAQ: AMZN), Tesla (NASDAQ: TSLA), and Apple (NASDAQ: AAPL) leading the way.'}, {'triplet_id': '111', 'entity': 'Tesla', 'entity_type': 'COMP', 'relationship': 'Positively_Impacts', 'target': 'top stocks', 'target_type': 'FIN_INST', 'date': '2023-12-01 00:00:00+00:00', 'ticker': 'AAPL', 'source_url': 'https://www.nasdaq.com/articles/should-you-really-invest-in-stocks-now-or-wait-until-the-new-year', 'source_text': 'top stocks such as Amazon (NASDAQ: AMZN), Tesla (NASDAQ: TSLA), and Apple (NASDAQ: AAPL) leading the way.'}, {'triplet_id': '112', 'entity': 'Apple', 'entity_type': 'ORG', 'relationship': 'Positively_Impacts', 'target': 'top stocks', 'target_type': 'FIN_INST', 'date': '2023-12-01 00:00:00+00:00', 'ticker': 'AAPL', 'source_url': 'https://www.nasdaq.com/articles/should-you-really-invest-in-stocks-now-or-wait-until-the-new-year', 'source_text': 'top stocks such as Amazon (NASDAQ: AMZN), Tesla (NASDAQ: TSLA), and Apple (NASDAQ: AAPL) leading the way.'}, {'triplet_id': '113', 'entity': 'December', 'entity_type': 'EVENT', 'relationship': 'Negatively_Impacts', 'target': 'S&P 500 index', 'target_type': 'FIN_MARKET', 'date': '2023-12-01 00:00:00+00:00', 'ticker': 'AAPL', 'source_url': 'https://www.nasdaq.com/articles/should-you-really-invest-in-stocks-now-or-wait-until-the-new-year', 'source_text': 'December brought the S&P 500 index a loss of about 5% last year'}, {'triplet_id': '114', 'entity': 'December', 'entity_type': 'EVENT', 'relationship': 'Positively_Impacts', 'target': 'S&P 500 index', 'target_type': 'FIN_MARKET', 'date': '2023-12-01 00:00:00+00:00', 'ticker': 'AAPL', 'source_url': 'https://www.nasdaq.com/articles/should-you-really-invest-in-stocks-now-or-wait-until-the-new-year', 'source_text': 'a gain of about as much in the previous year.'}, {'triplet_id': '115', 'entity': 'CMA', 'entity_type': 'ORG_REG', 'relationship': 'Involved_In', 'target': 'appeal from UK High Court', 'target_type': 'LITIGATION', 'date': '2023-12-01 00:00:00+00:00', 'ticker': 'AAPL', 'source_url': 'https://www.nasdaq.com/articles/cma-wins-appeal-from-uk-high-court-in-apple-case', 'source_text': 'The Competition and Markets Authority or CMA has won an appeal from UK High Court'}, {'triplet_id': '116', 'entity': 'CMA', 'entity_type': 'ORG_REG', 'relationship': 'Regulates', 'target': 'mobile browsers', 'target_type': 'PRODUCT', 'date': '2023-12-01 00:00:00+00:00', 'ticker': 'AAPL', 'source_url': 'https://www.nasdaq.com/articles/cma-wins-appeal-from-uk-high-court-in-apple-case', 'source_text': 'allowing it to open a market investigation into mobile browsers'}, {'triplet_id': '117', 'entity': 'CMA', 'entity_type': 'ORG_REG', 'relationship': 'Regulates', 'target': 'cloud gaming', 'target_type': 'PRODUCT', 'date': '2023-12-01 00:00:00+00:00', 'ticker': 'AAPL', 'source_url': 'https://www.nasdaq.com/articles/cma-wins-appeal-from-uk-high-court-in-apple-case', 'source_text': 'allowing it to open a market investigation into mobile browsers and cloud gaming'}, {'triplet_id': '118', 'entity': 'market investigation', 'entity_type': 'EVENT', 'relationship': 'Related_To', 'target': 'Apple Inc.', 'target_type': 'ORG', 'date': '2023-12-01 00:00:00+00:00', 'ticker': 'AAPL', 'source_url': 'https://www.nasdaq.com/articles/cma-wins-appeal-from-uk-high-court-in-apple-case', 'source_text': 'mainly of tech major Apple Inc. (AAPL).'}, {'triplet_id': '119', 'entity': 'The ruling', 'entity_type': 'LITIGATION', 'relationship': 'Negatively_Impacts', 'target': "Competition Appeal Tribunal or CAT's decision in March 2023", 'target_type': 'LITIGATION', 'date': '2023-12-01 00:00:00+00:00', 'ticker': 'AAPL', 'source_url': 'https://www.nasdaq.com/articles/cma-wins-appeal-from-uk-high-court-in-apple-case', 'source_text': "The ruling overturns the Competition Appeal Tribunal or CAT's decision in March 2023"}, {'triplet_id': '120', 'entity': "Competition Appeal Tribunal or CAT's decision", 'entity_type': 'LITIGATION', 'relationship': 'Positively_Impacts', 'target': 'appeal by Apple', 'target_type': 'LITIGATION', 'date': '2023-12-01 00:00:00+00:00', 'ticker': 'AAPL', 'source_url': 'https://www.nasdaq.com/articles/cma-wins-appeal-from-uk-high-court-in-apple-case', 'source_text': 'which upheld an appeal by Apple'}, {'triplet_id': '121', 'entity': "Competition Appeal Tribunal or CAT's decision", 'entity_type': 'LITIGATION', 'relationship': 'Negatively_Impacts', 'target': "CMA's investigation", 'target_type': 'EVENT', 'date': '2023-12-01 00:00:00+00:00', 'ticker': 'AAPL', 'source_url': 'https://www.nasdaq.com/articles/cma-wins-appeal-from-uk-high-court-in-apple-case', 'source_text': "and suspended CMA's investigation."}, {'triplet_id': '122', 'entity': 'Apple', 'entity_type': 'ORG', 'relationship': 'Involved_In', 'target': 'appealing to the CAT', 'target_type': 'LITIGATION', 'date': '2023-12-01 00:00:00+00:00', 'ticker': 'AAPL', 'source_url': 'https://www.nasdaq.com/articles/cma-wins-appeal-from-uk-high-court-in-apple-case', 'source_text': 'The lawfulness of that decision was challenged by Apple by appealing to the CAT'}, {'triplet_id': '123', 'entity': 'Apple Inc.', 'entity_type': 'ORG', 'relationship': 'Announces', 'target': "first and largest customer of Amkor Technology, Inc.'s new $2 billion advanced silicon manufacturing and packaging facility", 'target_type': 'COMP', 'date': '2023-12-01 00:00:00+00:00', 'ticker': 'AAPL', 'source_url': 'https://www.nasdaq.com/articles/apple-to-be-first-and-largest-customer-of-amkors-%242-bln-chip-packaging-plant', 'source_text': "Apple Inc. announced it will be the first and largest customer of Amkor Technology, Inc.'s new $2 billion advanced silicon manufacturing and packaging facility"}, {'triplet_id': '124', 'entity': 'Amkor Technology, Inc.', 'entity_type': 'COMP', 'relationship': 'Invests_In', 'target': 'new $2 billion advanced silicon manufacturing and packaging facility', 'target_type': 'PRODUCT', 'date': '2023-12-01 00:00:00+00:00', 'ticker': 'AAPL', 'source_url': 'https://www.nasdaq.com/articles/apple-to-be-first-and-largest-customer-of-amkors-%242-bln-chip-packaging-plant', 'source_text': "Amkor Technology, Inc.'s new $2 billion advanced silicon manufacturing and packaging facility being developed"}, {'triplet_id': '125', 'entity': 'new facility', 'entity_type': 'PRODUCT', 'relationship': 'Operates_In', 'target': 'Peoria, Arizona', 'target_type': 'GPE', 'date': '2023-12-01 00:00:00+00:00', 'ticker': 'AAPL', 'source_url': 'https://www.nasdaq.com/articles/apple-to-be-first-and-largest-customer-of-amkors-%242-bln-chip-packaging-plant', 'source_text': 'facility being developed in Peoria, Arizona.'}, {'triplet_id': '126', 'entity': 'new facility', 'entity_type': 'PRODUCT', 'relationship': 'Produces', 'target': 'Apple silicon', 'target_type': 'PRODUCT', 'date': '2023-12-01 00:00:00+00:00', 'ticker': 'AAPL', 'source_url': 'https://www.nasdaq.com/articles/apple-to-be-first-and-largest-customer-of-amkors-%242-bln-chip-packaging-plant', 'source_text': 'The new facility, which is expected to employ around 2,000 people upon completion, will package Apple silicon'}, {'triplet_id': '127', 'entity': 'Apple silicon', 'entity_type': 'PRODUCT', 'relationship': 'Related_To', 'target': 'Taiwan Semiconductor Manufacturing Co Ltd. or TSMC fab', 'target_type': 'COMP', 'date': '2023-12-01 00:00:00+00:00', 'ticker': 'AAPL', 'source_url': 'https://www.nasdaq.com/articles/apple-to-be-first-and-largest-customer-of-amkors-%242-bln-chip-packaging-plant', 'source_text': 'Apple silicon produced at the nearby Taiwan Semiconductor Manufacturing Co Ltd. or TSMC fab'}, {'triplet_id': '128', 'entity': 'Apple', 'entity_type': 'ORG', 'relationship': 'Depends_On', 'target': 'Taiwan Semiconductor Manufacturing Co Ltd. or TSMC fab', 'target_type': 'COMP', 'date': '2023-12-01 00:00:00+00:00', 'ticker': 'AAPL', 'source_url': 'https://www.nasdaq.com/articles/apple-to-be-first-and-largest-customer-of-amkors-%242-bln-chip-packaging-plant', 'source_text': 'where Apple is also the largest customer.'}, {'triplet_id': '129', 'entity': 'Apple', 'entity_type': 'ORG', 'relationship': 'Invests_In', 'target': '$430 billion in the U.S. economy', 'target_type': 'ORG_GOV', 'date': '2023-12-01 00:00:00+00:00', 'ticker': 'AAPL', 'source_url': 'https://www.nasdaq.com/articles/apple-to-be-first-and-largest-customer-of-amkors-%242-bln-chip-packaging-plant', 'source_text': 'Apple in 2021 had committed to invest $430 billion in the U.S. economy over five years.'}]
    
    # 3. Add relationships to Neo4j Instance
    load_dotenv()
    graph = Neo4jGraph(
        url=os.getenv('NEO4J_URI', ''),
        username=os.getenv('NEO4J_USERNAME', ''),
        password=os.getenv('NEO4J_PASSWORD', ''),
        database=os.getenv('NEO4J_DATABASE', '')
    )
    # clear_kg(graph)
    # add_relationship_to_neo4j(graph, triplets)
    # visualize_kg(graph)
    
    # 4. LLMs output from few-shot guiding questions (10 questions)
    questions = [
        "What products does Apple produce?",
        "How much did AAPL stock increase?",
        "Who are Apple's competitors?",
        "What events affected Apple's stock?",
        "What products does Apple produce and which events increased their sales?",
        "How many relationships does Apple have in the graph?",
        "What happened with Apple in December 2023?"
    ]
    results_filename = 'results.csv'
    for question in questions:
        response = query_chatbot(
            question=question,
            neo4j_session=graph
        )
        query_to_csv(response, results_filename)
    graph.close()
    
    # 5. Run LLM as a judge on input and output, compare with groundtruth answer done by human